## Configuration Generator for Oracle
This notebook facilitates the creation of one or multiple configurations in a single run. Follow the guidelines given below.

Initially, you should execute the entire notebook to make the widgets visible. Keep filling in the necessary information in the widgets and rerun the notebook until all widgets are populated.

#### Input Reference:

1. **Connection String**: Please input the connection string that's stored in your key-vault.
2. **Destination for Configurations**: Choose the target location for your configuration files. Ensure that the designated configuration folder is created before you start the generation process.
3. **Bronze Load Type**: Here, you can select from the available load types. Please note that any overwriting operation on a bronze table with more than 100,000 records will automatically switch to append mode.
4. **Silver Load Type**: Similarly, you can select from the available load types for Silver. We suggest choosing the 'Upsert' method while defining the primary keys.
5. **Schema Selection**: Once you provide a connection string, this option will become available. Changing the schema will automatically update the dropdown list for table selection.
6. **Table Selection**: This accepts the same expressions as an `SQL LIKE` statement (e.g., `PPS%`). Use `%` to select all tables in a schema (exercise caution). For partial matches, use: `PPS%`. Note: Even if a table doesn't appear in the dropdown, you can still enter its name as we allow free form input. However, the dropdown is capped at 1024 items, so if your schema has more tables, you will have to input them manually.
7. **Source/Bronze Watermark**: Please provide a regular expression to identify a column to act as a watermark. If there are multiple matches, the system will select the one that appears closest to the end of the table definition in the information schema.
8. **Bronze/Silver Watermark**: If set to *None*, no watermark will be established, triggering a full read from the bronze data layer. If set to *__created_tsp*, only newly arrived records in the bronze layer will be considered when transferring data to the silver layer. We highly recommend this setting for efficiency.
9. **Additional Options**: The 'Set Nullable' option updates each column's nullable attribute based on what's recorded in the Oracle information schema. The 'Set Primary Keys' option sets each column's metadata key attribute to 'true' if that column in Oracle is constrained by a Primary or Unique Key. The 'Get Snowflake Comment' will try and retrieve a comment from Snowflake for a similar named field, if no source comment is present.
10. **Set Secret Scope**: The default secret scope for most procedures is "alchemy-kv" but there are some secure sources that require a different scope to access the key vault in Azure

In [0]:
import os, sys

import data.utilities.common as Commonlib
import yaml
from beertech_utils.core.logger import LogManager
from data.utilities.extractors import OracleExtractor, SnowflakeExtractor
from pyspark.sql.functions import desc, expr
from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    ByteType,
    DateType,
    DecimalType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    ShortType,
    StringType,
    TimestampType,
)

In [0]:
OPTION_SNOWFLAKE_COMMENT = "Get Snowflake Comment"
OPTION_PRIMARY_KEYS = "Set Primary Keys"
OPTION_NULLABLE = "Set Nullable"

CONFIG_BASE_PATH = "../../configuration/oracle/"
BRONZE_LOAD_TYPES = ["append", "overwrite"]
SILVER_LOAD_TYPES = ["append", "overwrite", "upsert"]
BOOLEAN_OPTIONS = ["yes", "no"]
COMMENT_OPTIONS = ["Source, FieldName"]
BRONZE_WATERMARK_OPTIONS = ["None", ".*MOD.*TSP$", ".*MDFD.*TSP$"]
SILVER_WATERMARK_OPTIONS = ["None", "__created_tsp"]
OTHER_OPTIONS = [OPTION_NULLABLE, OPTION_PRIMARY_KEYS, OPTION_SNOWFLAKE_COMMENT]
SECRET_SCOPE_OPTIONS = ["alchemy-kv", "people-kv"]
MAX_ROWS_ALLOWED_OVERWRITE = 100000

# Configure the logging module
logger = LogManager.get_logger("Oracle Configuration Generator")

#### get value from all widgets that should not cause oracle queries to execute
# subject areas available for configurations - configs will be created in this location
subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "2. Subject Area for Configs")
subject_area = dbutils.widgets.get("subject_area")

# set load types available
dbutils.widgets.dropdown("bronze_load_type", "append", BRONZE_LOAD_TYPES, "3. Bronze Load Type")
bronze_load_type = dbutils.widgets.get("bronze_load_type")
dbutils.widgets.dropdown("silver_load_type", "upsert", SILVER_LOAD_TYPES, "4. Silver Load Type")
silver_load_type = dbutils.widgets.get("silver_load_type")

# display available schemas
dbutils.widgets.multiselect("other_options", "", [""] + OTHER_OPTIONS, "9. Other Options")
other_selections = dbutils.widgets.get("other_options")
other_selections = other_selections.split(",")

# source/bronze watermark logic
dbutils.widgets.combobox("bronze_watermark", "None", BRONZE_WATERMARK_OPTIONS, "7. Source/Bronze Watermark")
bronze_watermark = dbutils.widgets.get("bronze_watermark")

# bronze/silver watermark logic
dbutils.widgets.combobox("silver_watermark", "None", SILVER_WATERMARK_OPTIONS, "8. Bronze/Silver Watermark")
silver_watermark = dbutils.widgets.get("silver_watermark")

# set secret scope logic
dbutils.widgets.dropdown("secret_scope", "alchemy-kv", SECRET_SCOPE_OPTIONS, "10. Secret Scope")
secret_scope = dbutils.widgets.get("secret_scope")

In [0]:
#### get value from widgets that should cause the schema drop-down query to execute
# connection string input
dbutils.widgets.text("connection", "", "1. Connection String")

# setup oracle extractor via connection string
connection = dbutils.widgets.get("connection").lower()
oracle_extractor = OracleExtractor(secret_scope=secret_scope, secret_key=connection)

# get available schemas
available_schemas_query = "select distinct owner from sys.all_tables where tablespace_name not in ('SYSTEM','SYSAUX','SNAPDATA','DBATOOLS','APP_UTIL_DATA') order by 1 asc"
available_schemas = oracle_extractor.query(available_schemas_query)
available_schemas = [row.OWNER.lower() for row in available_schemas.select("OWNER").collect()]

# display available schemas
dbutils.widgets.dropdown("oracle_schema", "", [""] + available_schemas, "5. Select Schema")

In [0]:
#### get value from widgets that should cause a table dropdown refresh
# get value from schema
oracle_schema = dbutils.widgets.get("oracle_schema")

# get available tables for specified schema
available_tables_query = (
    f"select distinct TABLE_NAME from sys.all_tab_columns where OWNER=UPPER('{oracle_schema}') order by 1 asc"
)
available_tables = oracle_extractor.query(available_tables_query)

# display available tables
available_tables = [row.TABLE_NAME.lower() for row in available_tables.select("TABLE_NAME").collect()]

# truncate list to 1024 values
if len(available_tables) > 1023:
    del available_tables[1022:]
    logger.warning(
        f"The schema, {oracle_schema}, contains more than 1024 values; Truncating to show only 1024 values if your table is not listed you will have to manually type it in the '6. Select Table' textbox."
    )

dbutils.widgets.combobox("oracle_table", "", [""] + available_tables, "6. Select Table")

In [0]:
# get selected table
oracle_table = dbutils.widgets.get("oracle_table").upper()

# if blank set value to get all tables in schema
# oracle_table = oracle_table.upper()

logger.info(f"Tables to generate: {oracle_table}")

# get detailed data from oracle information schema
with open("oracle_generator_info_schema.sql", "r") as file:
    oracle_info_schema_query = file.read()

oracle_info_schema_query = oracle_info_schema_query.format(oracle_schema=oracle_schema, oracle_table=oracle_table)

In [0]:
#### snowflake comment retrieval
# determines if snowflake comment option is selected and whether the comment_lookup variable is already in memory - to execute a fresh query against snwoflake detach and re-attach the notebook
if OPTION_SNOWFLAKE_COMMENT in other_selections and "comment_lookup" not in globals():
    # get column comments from snowflake information schema
    with open("snowflake_comments.sql", "r") as file:
        snowflake_query = file.read()

    # get production comment information
    snowflake_extract = SnowflakeExtractor("ABI_WH", "PUBLIC")

    all_snowflake_column_desc = snowflake_extract.query(snowflake_query)

    # create dictionary comment lookup
    rows = all_snowflake_column_desc.select("COLUMN_NAME", "COMMENT").collect()
    comment_lookup = {row["COLUMN_NAME"].lower(): row["COMMENT"] for row in rows}

In [0]:
info_schema_df = oracle_extractor.query(oracle_info_schema_query)

# cast to correct data types
info_schema_df = info_schema_df.withColumn("DATA_SCALE", info_schema_df["DATA_SCALE"].cast("integer"))
info_schema_df = info_schema_df.withColumn("DATA_PRECISION", info_schema_df["DATA_PRECISION"].cast("integer"))
info_schema_df = info_schema_df.withColumn("NUM_ROWS", info_schema_df["NUM_ROWS"].cast("integer"))

In [0]:
#### helpers
# oracle data types mapping for pyspark
oracle_to_pyspark = {
    "VARCHAR2": StringType(),
    "NVARCHAR2": StringType(),
    "CHAR": StringType(),
    "NCHAR": StringType(),
    "CLOB": StringType(),
    "NCLOB": StringType(),
    "BLOB": BinaryType(),
    "RAW": BinaryType(),
    "NUMBER": DecimalType(38, 10),
    "FLOAT": DoubleType(),
    "BINARY_FLOAT": FloatType(),
    "BINARY_DOUBLE": DoubleType(),
    "DATE": TimestampType(),
    "TIMESTAMP": TimestampType(),
    "TIMESTAMP WITH TIME ZONE": TimestampType(),
    "TIMESTAMP WITH LOCAL TIME ZONE": TimestampType(),
    "INTERVAL YEAR TO MONTH": StringType(),
    "INTERVAL DAY TO SECOND": StringType(),
    "LONG": StringType(),
    "LONG RAW": BinaryType(),
    "ROWID": StringType(),
    "UROWID": StringType(),
    "INTEGER": IntegerType(),
    "SMALLINT": ShortType(),
    "REAL": DoubleType(),
    "DOUBLE PRECISION": DoubleType(),
    "BOOLEAN": BooleanType(),
}

# metadata columns - currently used in silver for delta loads from bronze
default_created_tsp = {
    "name": "__created_tsp",
    "type": "timestamp",
    "nullable": False,
    "metadata": {"comment": "Timestamp indicating the time at which the record was originally created in delta table."},
}

default_source = {
    "name": "__source",
    "type": "string",
    "nullable": False,
    "metadata": {"comment": "Metadata field indicating the source of the record."},
}


# generates a schema field
def generate_field(row, other_options):
    """
    This function generates a dictionary representing a schema field.

    The function takes a Row object as input, which contains metadata about a column
    (like its name, type, nullable property, etc.) and returns a dictionary
    that represents the schema field for that column.

    Args:
        row (pyspark.sql.Row): A Row object containing metadata about a column.

    Returns:
        dict: A dictionary representing the schema field for the column.
    """
    col_data_precision = row.DATA_PRECISION
    col_data_scale = row.DATA_SCALE
    col_comment = row.COLUMN_COMMENTS.replace("\n", " ") if row.COLUMN_COMMENTS else row.COLUMN_NAME.lower()

    if col_comment == row.COLUMN_NAME.lower() and OPTION_SNOWFLAKE_COMMENT in other_options:
        col_comment = comment_lookup.get(row.COLUMN_NAME.lower(), col_comment)

    # data type handling for special cases
    if row.DATA_TYPE == "NUMBER":
        # sometimes scale and precision for a number type are blank or when scale is zero precision is 38
        if col_data_precision is None and (col_data_scale is None or col_data_scale == 0):
            col_data_precision = 38
            col_data_scale = 0

        field = {"name": row.COLUMN_NAME.lower(), "type": f"decimal({col_data_precision}, {col_data_scale})"}
    else:
        field = {"name": row.COLUMN_NAME.lower(), "type": oracle_to_pyspark[row.DATA_TYPE].typeName()}

    # determine if field is nullable - only set if false - default is true
    if row.NULLABLE == "N" and OPTION_NULLABLE in other_options:
        field["nullable"] = False

    # generate metadata dict
    field_metadata = {"metadata": {"comment": col_comment}}

    # determine if column was part of primary/unique key constraint
    if row.UNIQUE_KEY == "Y" and OPTION_PRIMARY_KEYS in other_options:
        field_metadata.get("metadata")["key"] = True

    return {**field, **field_metadata}


# generates a configuration for an entire table
def process_table(table_name, table_df, bronze_load_type=bronze_load_type, other_options=None):
    """
    This function generates a configuration dictionary for a table.

    The function takes a table name, a DataFrame containing metadata about the table,
    and a load type as input. It generates a configuration dictionary for the table,
    which includes its schema and other properties.

    The function also handles several special cases, such as large data sets and
    watermarks for bronze and silver data.

    Args:
        table_name (str): The name of the table.
        table_df (pyspark.sql.DataFrame): A DataFrame containing metadata about the table.
        bronze_load_type (str): The load type for the bronze data.

    Returns:
        dict: A dictionary representing the configuration for the table.
    """
    schema = []
    config = {}
    bronze_watermark_col = ""

    # Get the first record for current table_name
    first_record = table_df.first()

    # prevent overwrite into bronze if data set is too large
    if (
        first_record.NUM_ROWS is not None
        and first_record.NUM_ROWS > MAX_ROWS_ALLOWED_OVERWRITE
        and bronze_load_type == "overwrite"
    ):
        bronze_load_type = "append"
        logger.warning(
            f"Load type for {table_name} was changed from overwrite to append. Max records reached for overwrite operations into bronze."
        )

    # start building config for table
    config = {
        "version": "1.0.0",
        "name": table_name.lower(),
        "schema": first_record.OWNER.lower(),
        "catalog": "oracle",
        "description": first_record.TABLE_COMMENTS if first_record.TABLE_COMMENTS else table_name.lower(),
        "source": {"connection": connection.lower(), "schema": first_record.OWNER.lower()},
        "bronze": {"load_type": bronze_load_type},
        "silver": {"load_type": silver_load_type},
    }

    if secret_scope != "alchemy-kv":
        config["source"]["secret_scope"] = secret_scope.lower()

    # determine bronze watermark
    if bronze_watermark != "None":
        bronze_watermark_df = table_df.filter(
            (table_df.DATA_TYPE == "DATE") | (table_df.DATA_TYPE == "TIMESTAMP")
        ).filter(table_df["COLUMN_NAME"].rlike(bronze_watermark))
        # get best match for bronze watermark
        if bronze_watermark_df.count() > 0:
            # sort by column id - we want the match closes to the end
            bronze_watermark_df = bronze_watermark_df.sort(desc("COLUMN_ID"))
            bronze_watermark_df = bronze_watermark_df.first()
            bronze_watermark_col = bronze_watermark_df["COLUMN_NAME"]
            config["bronze"]["watermark"] = bronze_watermark_col.lower()
        else:
            config["bronze"]["watermark"] = "auto_find_failed"

    # set silver watermark
    if silver_watermark == "Same as bronze" and bronze_watermark_col:
        config["silver"]["watermark"] = bronze_watermark_col.lower()
    elif silver_watermark == "__created_tsp":
        config["silver"]["watermark"] = "__created_tsp"

    # loop through every row/column to genrate the schema field
    for row in table_df.collect():
        schema.append(generate_field(row, other_options))

    # add default columns
    schema.append(default_created_tsp)
    schema.append(default_source)

    config["silver"]["schema"] = schema

    return config


#### generate configurations
# confirm subject area is set - prevents massive amount of files from being generated at configuration/oracle root folder
if not subject_area:
    raise ValueError("Subject area must be set in order to continue.")

if not oracle_table:
    raise ValueError("Select or provide table name in order to continue.")

# get tables that need configurations
table_names = [row.TABLE_NAME for row in info_schema_df.select("TABLE_NAME").distinct().sort("TABLE_NAME").collect()]

# loop through all the tables and generate config file in proper folder
for table_name in table_names:
    logger.info(f"Processing table: {table_name}")

    table_df = info_schema_df.filter(info_schema_df.TABLE_NAME == table_name).sort(info_schema_df.COLUMN_ID.asc())

    config = process_table(table_name, table_df, other_options=other_selections)
    config_write_path = f"{CONFIG_BASE_PATH}{subject_area}"
    config_file = f"{table_name.lower()}.yml"

    with open(os.path.join(config_write_path, config_file), "w") as outfile:
        yaml.dump(config, outfile, indent=2, sort_keys=False)